#Step -1 : Create a Volume to Stimulate incoming files

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS insurance_dev.raw_files.claims_stream;




Step 2 — Adapted lab code (paths changed to Volumes)


In [0]:
import json 

# Stimulate a new claim file landing ( like a source system dropping a file)
sample_claim = {"claim_id":"CML100", "policy_id":"POL1001","claim_amount":"5000","claim_status":"Filed"}

with open("/Volumes/insurance_dev/raw_files/claims_stream/claim_batch1.json", "w")  as f:
    f.write(json.dumps(sample_claim))

# Auto Loader - Bronze ingestion (schema location also in a volume, not /tmp)
df_bronze_stream = (spark.readStream
    .format("cloudFiles")                    
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/insurance_dev/raw_files/claims_stream/_schema")
    .load("/Volumes/insurance_dev/raw_files/claims_stream/")
    )

# Auto Loader - Silver ingestion (schema location also in a volume, not /tmp)
(df_bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/insurance_dev/raw_files/claims_stream/_checkpoint")
    .trigger(availableNow=True)
    .toTable("insurance_dev.raw_files.claims_bronze")
    )

display(spark.sql("SELECT * FROM insurance_dev.raw_files.claims_bronze"))




claim_amount,claim_id,claim_status,policy_id,_rescued_data
